# Lab 9: Do Optimized Portfolios Beat Naive Ones?

*Financial Engineering in Practice*, Chapter 9.

**What you will explore:** how optimized portfolio strategies compare with equal weighting (1/N) out of sample.

You will run allocation rules walk-forward on industry and size/book-to-market portfolios, charge trading costs, and look at why optimized weights struggle. Before each step, write down a prediction.

`FAST = True` runs six rules in about a minute; set it to `False` for all nine (about five minutes).

In [ ]:
from pathlib import Path
from IPython.display import Image, display
import lab09
from fintest.portfolio import RULES

SOURCE = "french"     # "french" downloads from the Kenneth French library; "sim" uses fake data to test the code
FILES = {}            # optional: {"ff3": ..., "ind10": ..., "ind30": ..., "sbm25": ...} paths to saved zip files
FAST = True
WINDOW, COST = 120, 0.0025
OUT = Path("report")
OUT.mkdir(exist_ok=True)

d = lab09.load(SOURCE, FILES)
SIM = d["simulated"]
NAMES = lab09.FAST if FAST else list(RULES)
LABEL, R = next(iter(d["sets"].items()))
print({k: v.shape for k, v in d["sets"].items()}, "| simulated:", SIM)

## Part A: The rules on ten industries

**Predict first:** with ten years of data to estimate means and covariances, will sample mean-variance beat 1/N out of sample? Will minimum variance?

In [ ]:
a = lab09.part_a(R, WINDOW, NAMES)
lab09.fig_sharpe(a, LABEL, OUT, SIM)
lab09.fig_turnover(a, OUT, SIM)
display(Image(OUT / "fig9_1_sharpe.png"))
display(Image(OUT / "fig9_2_turnover.png"))
a["table"].round(3)

## Part B: Other asset sets, after costs

**Predict first:** with 30 or 25 assets, does optimization get better or worse relative to 1/N?

In [ ]:
b = lab09.part_b(d["sets"], WINDOW, NAMES, COST, first_res=a["res"])
lab09.fig_heatmap(b, OUT, SIM)
display(Image(OUT / "fig9_3_heatmap.png"))
b.round(3)

## Part C: Stress tests

**Predict first:** does a longer estimation window rescue mean-variance? Does rebalancing less often help after costs?

In [ ]:
fast = [n for n in NAMES if n in lab09.FAST]
c = lab09.part_c(R, fast, {n: a["res"][n] for n in fast})
lab09.fig_windows(c, OUT, SIM)
display(Image(OUT / "fig9_4_windows.png"))
display(c["costs"].round(3))
display(c["rebalance"].round(3))
c["crisis"].round(3)

## Part D: Why — weight instability

In [ ]:
dd = lab09.part_d(a["res"], R)
lab09.fig_weights(a["res"], R, OUT, SIM)
display(Image(OUT / "fig9_5_weights.png"))
dd.round(3)

## Questions to think about

1. Did any optimized rule beat 1/N by a margin the Sharpe-difference test can distinguish from noise?
2. Which rules survived costs? Which ones relied on high turnover?
3. Is the problem the optimizer or the inputs?

Write one paragraph on what you saw, then compare it with the chapter's section Where the evidence stands.